# From Normal Distributions to Bayesian Optimization
### MZ | AIchemy Winter School: Robotics and AI for chemistry, 23 November, 2026

**Together (sections 0–7): 35 minutes.** We will edit together; all code starts runnable.

**After class (A–F): optional exploration**, including kernel comparisons, higher-dimensional optimization, and scikit-learn, BoTorch and BayBE examples.

**One consistent chemistry example:** Maximize a synthetic reaction yield over reaction temperatures from 60–140°C, starting with three observations at 65, 90 and 130°C. Measurement noise has standard deviation 2 percentage points; random seed = 2026.

**Learning goal:** Build the *concepts* of BO from scratch.

> Note: the unknown true reaction curve is a simulation, not real chemistry data. The model does not see it until an experiment is requested. Raw GP Gaussian predictions need not obey physical bounds such as 0–100% yield.

---

## 0. Setup · 2 min

Run the cell below. It includes numerical and plotting helpers, the hidden synthetic reaction and fixed observations.

**You do not have to type the helper functions.**

In [ ]:
# @title Setup and plotting helpers (run only; no editing) { display-mode: "form" }
"""Standalone teaching helpers embedded into the Colab notebooks at build time
No GP/BO libraries needed in the live session
"""
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SEED=2026
NOISE_STD=2.0
PRIOR_MEAN=50.0
PRIOR_SD=18.0
X_GRID=np.linspace(60,140,201)
INITIAL_X=np.array([65.0,90.0,130.0])

def true_yield(x):
    x=np.asarray(x,dtype=float)
    return (23.0+37.0*np.exp(-.5*((x-83.0)/10)**2)
            +54.0*np.exp(-.5*((x-116.0)/7)**2)
            +3.0*np.sin((x-60.0)/7))

_rng=np.random.default_rng(SEED)
INITIAL_Y=true_yield(INITIAL_X)+_rng.normal(0,NOISE_STD,len(INITIAL_X))

def as_2d(x):
    x=np.asarray(x,dtype=float)
    if x.ndim==0:return x.reshape(1,1)
    if x.ndim==1:return x[:,None]
    return x

def kernel_matrix(x,xp,kind='RBF',ell=15.0,amplitude=18.0,period=25.0):
    a,b=as_2d(x),as_2d(xp)
    assert a.shape[1]==b.shape[1]
    ell=np.asarray(ell,dtype=float)
    if np.any(ell<=0):raise ValueError('ell must be positive')
    d=(a[:,None,:]-b[None,:,:])/ell
    r2=np.sum(d*d,axis=-1)
    r=np.sqrt(r2)
    if kind=='RBF':z=np.exp(-.5*r2)
    elif kind=='Matern 3/2':z=(1+np.sqrt(3)*r)*np.exp(-np.sqrt(3)*r)
    elif kind=='Matern 5/2':z=(1+np.sqrt(5)*r+5*r2/3)*np.exp(-np.sqrt(5)*r)
    elif kind=='Linear':z=1+((a-100)/40)@((b-100)/40).T
    elif kind=='Periodic':
        if a.shape[1]!=1:raise ValueError('Periodic helper supports 1 input')
        delta=np.abs(a[:,None,0]-b[None,:,0])
        z=np.exp(-2*np.sin(np.pi*delta/period)**2/ell**2)
    else:raise ValueError(f'Unknown kernel: {kind}')
    return amplitude**2*z

def sample_prior(x,kind='RBF',ell=15,amplitude=18,mean=50,n=4,seed=2026):
    K=kernel_matrix(x,x,kind,ell,amplitude)
    L=np.linalg.cholesky(K+1e-8*np.eye(len(x)))
    return mean+(L@np.random.default_rng(seed).standard_normal((len(x),n))).T

def gp_predict(x_obs,y_obs,x_test,kind='RBF',ell=15,
               amplitude=18,mean=50,noise_std=2):
    """Exact latent GP mean and standard deviation with Gaussian observation noise."""
    y=np.asarray(y_obs,dtype=float).ravel()
    K=kernel_matrix(x_obs,x_obs,kind,ell,amplitude)
    Ks=kernel_matrix(x_obs,x_test,kind,ell,amplitude)
    Kss=kernel_matrix(x_test,x_test,kind,ell,amplitude)
    L=np.linalg.cholesky(K+(noise_std**2+1e-8)*np.eye(len(y)))
    alpha=np.linalg.solve(L.T,np.linalg.solve(L,y-mean))
    v=np.linalg.solve(L,Ks)
    mu=mean+Ks.T@alpha
    var=np.maximum(np.diag(Kss)-np.sum(v*v,axis=0),0)
    return mu,np.sqrt(var)

def plot_gp(x_obs,y_obs,ell=15,kind='RBF',kappa=None,
            show_truth=False,title='Gaussian Process'):
    mu,sd=gp_predict(x_obs,y_obs,X_GRID,kind,ell,PRIOR_SD,PRIOR_MEAN,NOISE_STD)
    if kappa is None:
        fig=go.Figure()
        target=(1,1)
    else:
        fig=make_subplots(rows=2,cols=1,shared_xaxes=True,
                 row_heights=[.63,.37],vertical_spacing=.12,
                 subplot_titles=['Posterior mean and uncertainty','Acquisition score'])
        target=(1,1)
    def add(trace,row=1):
        if kappa is None:fig.add_trace(trace)
        else:fig.add_trace(trace,row=row,col=1)
    add(go.Scatter(x=X_GRID,y=mu-2*sd,line=dict(width=0),
                   showlegend=False,hoverinfo='skip'))
    add(go.Scatter(x=X_GRID,y=mu+2*sd,line=dict(width=0),
                   fill='tonexty',fillcolor='rgba(55,124,185,0.20)',
                   name='Approx. 95% latent interval'))
    add(go.Scatter(x=X_GRID,y=mu,line=dict(color='#397CB9',width=3),
                   name='Posterior mean'))
    add(go.Scatter(x=x_obs,y=y_obs,mode='markers',
                   marker=dict(color='#D66C43',size=10),name='Measured yields'))
    if show_truth:
        add(go.Scatter(x=X_GRID,y=true_yield(X_GRID),line=dict(color='#555',dash='dash'),name='Hidden truth'))
    if kappa is not None:
        ucb=mu+kappa*sd
        idx=int(np.argmax(ucb))
        add(go.Scatter(x=X_GRID,y=ucb,line=dict(color='#239267',width=3),
                       name=f'UCB: κ={kappa:g}'),row=2)
        add(go.Scatter(x=[X_GRID[idx]],y=[ucb[idx]],mode='markers',
                       marker=dict(size=12,color='#CF913A'),
                       name=f'Next experiment: {X_GRID[idx]:.1f}°C'),row=2)
        for row in [1,2]:fig.add_vline(x=float(X_GRID[idx]),line_color='#CF913A',
                                        line_dash='dash',row=row,col=1)
    fig.update_layout(title=title,template='plotly_white',
                      height=650 if kappa is not None else 440,
                      legend=dict(orientation='h',y=1.12),
                      margin=dict(l=65,r=40,t=110,b=55))
    if kappa is None:
        fig.update_xaxes(title_text='Temperature (°C)',range=[60,140])
        fig.update_yaxes(title_text='Yield (%)',range=[0,105])
    else:
        fig.update_xaxes(title_text='Temperature (°C)',row=2,col=1,range=[60,140])
        fig.update_yaxes(title_text='Yield (%)',row=1,col=1,range=[0,105])
        fig.update_yaxes(title_text='UCB score',row=2,col=1)
    return fig

print("Initial temperatures (°C):",INITIAL_X)
print("Initial measured yields (%):",np.round(INITIAL_Y,1))

---

## 1. One uncertain value: normal distribution · 3 min

Suppose we have not measured the reaction yield at 90°C. A **normal distribution** describes possible outcomes at that one condition: $f(90)\sim\mathcal N(\mu,\sigma^2)$.


**TODO 1:** Change `std_yield` from 10 to 5, then 20. What changes? The centre, width, or both?

In [ ]:
from scipy.stats import norm
mean_yield = 50.0
std_yield = 10.0  # TODO 1: try 5 and 20
rng = np.random.default_rng(SEED)
sampled_yields = rng.normal(mean_yield, std_yield, 600)
xs = np.linspace(0, 100, 300)
fig=go.Figure()
fig.add_trace(go.Histogram(x=sampled_yields,histnorm='probability density',
              opacity=.45,nbinsx=35,name='600 possible yields'))
fig.add_trace(go.Scatter(x=xs,y=norm.pdf(xs,mean_yield,std_yield),
              line=dict(width=3),name='Normal distribution'))
fig.update_layout(title='One temperature: one uncertain yield',
    xaxis_title='Possible yield at 90°C (%)',yaxis_title='Probability density',
    template='plotly_white',height=410,barmode='overlay')
fig.show()

---

## 2. Two uncertain values: a joint Gaussian · 4 min

Suppose both $f(90)$ and $f(95)$ are unknown. The model can say whether these two values *vary together across plausible functions*. That is correlation/covariance.

`rho` is the correlation coefficient

**TODO 2:** Set `rho` to 0, 0.5, 0.95, -0.90, -0.50. What does the scatter cloud tell you?

In [ ]:
rho=0.90   # TODO 2: try 0.0, 0.5, 0.95, -0.90, -0.50
rng=np.random.default_rng(SEED)
z1,z2=rng.standard_normal((2,300))
yield_90=50+12*z1  # X = \mu + \sigma * Z, transfers a standard normal variable Z into N(\mu, \sigma^2)
yield_95=50+12*(rho*z1+np.sqrt(1-rho*rho)*z2)  # cholesky decomposition
fig=go.Figure(go.Scatter(x=yield_90,y=yield_95,mode='markers',
           marker=dict(size=6,color='#397CB9',opacity=.60)))
fig.update_layout(title=f'Two correlated yield predictions: ρ = {rho:.2f}',
    xaxis_title='Possible yield at 90°C (%)',
    yaxis_title='Possible yield at 95°C (%)',
    xaxis=dict(range=[5,95]),yaxis=dict(range=[5,95],scaleanchor='x'),
    template='plotly_white',height=460)
fig.show()

### The conceptual jump: from a normal variable to a Gaussian Process

- One experimental condition → one normally distributed unknown yield.
- Two conditions → a **joint two-dimensional** normal distribution of two possible yields.
- Many conditions → a **joint multivariate** normal distribution of all those possible yields.
- If we sample the jointly distributed values at many temperatures, we get **one possible function**. Repeat to obtain a distribution over functions.

$$f\sim\mathcal{GP}(m,k), \qquad m(x)=\mathbb E[f(x)],\qquad k(x,x')=\operatorname{Cov}(f(x),f(x')).$$


**Why does the kernel take x and x′, but tell us about f(x) and f(x′)?**

Because the kernel encodes a *prior modelling assumption*: similarity between experimental conditions determines how strongly their still-unknown responses are expected to vary together. It does **not** calculate a covariance from already measured yields.

---

## 3. Kernel → covariance → possible functions · 6 min

For an RBF covariance function, nearby temperatures are highly correlated. Its length scale controls how quickly correlation decays with separation:

$$k(x,x')=\sigma_f^2\exp\left[-\frac{(x-x')^2}{2\ell^2}\right].$$

- The heatmap is correlation, $k/\sigma_f^2$.

- The curves on the right are **prior samples**, before data.

**TODO 3:** Try `length_scale=6`, then `40`. Predict what will happen to the heatmap **and** the curves.

In [ ]:
length_scale=15.0  # TODO 3: try 6 and 40 (°C)
t=np.linspace(60,140,85)
K=kernel_matrix(t,t,'RBF',length_scale,PRIOR_SD)
curves=sample_prior(t,'RBF',length_scale,PRIOR_SD,PRIOR_MEAN,n=4,seed=SEED)
implied_rho = np.exp(-0.5*(5.0/length_scale)**2)
pairs=sample_prior(np.array([90.0,95.0]),'RBF',length_scale,PRIOR_SD,PRIOR_MEAN,n=1000,seed=SEED)
print(f'Kernel-implied correlation at 90°C and 95°C: {implied_rho:.3f}')
print(f'Correlation across 1000 prior-drawn yield pairs: {np.corrcoef(pairs.T)[0,1]:.3f}')
fig=make_subplots(rows=1,cols=2,horizontal_spacing=.14,
     subplot_titles=['Correlation: temperature pairs','Four possible yield curves'])
fig.add_trace(go.Heatmap(x=t,y=t,z=K/PRIOR_SD**2,  # convert covariance into correlation
    colorscale='Blues',zmin=0,zmax=1,
    colorbar=dict(title='Correlation',x=.43,len=.70)),row=1,col=1)
for j,curve in enumerate(curves):
    fig.add_trace(go.Scatter(x=t,y=curve,mode='lines',
        name=f'Possible function {j+1}',line=dict(width=2)),row=1,col=2)
fig.add_hline(y=PRIOR_MEAN,line_dash='dot',line_color='#888',row=1,col=2)
fig.update_layout(title=f'GP prior, RBF length scale {length_scale:g}°C',
                  height=490,width=1080,template='plotly_white',
                  legend=dict(orientation='h',y=-.22))
fig.update_xaxes(title_text='Temperature (°C)',row=1,col=1)
fig.update_yaxes(title_text='Temperature (°C)',row=1,col=1)
fig.update_xaxes(title_text='Temperature (°C)',row=1,col=2)
fig.update_yaxes(title_text='Prior yield (%)',row=1,col=2)
fig.show()

**Note**: the slight difference b/t *kernel-implied correlation* and
*correlation across 1000 prior-drawn yield pairs* happens because one is the exact theoretical value, the other is a random sample estimate based o a finite number of draws. As we increase number of pairs, these two values will converge. You can check yourself :)

---

## 4. Experiments → GP posterior · 6 min

We have three observed yields. A GP combines its prior assumptions with the data to produce the
- **posterior mean** $\mu(x)$ and
- **posterior latent standard deviation** $\sigma(x)$

The shaded interval is approximately $\mu(x)\pm2\sigma(x)$, expressing uncertainty about the *underlying yield function*, not the scatter of repeated noisy measurements.

**TODO 4:** Predict where uncertainty will be smallest. Optionally uncomment two lines to add a fourth simulated measurement.

In [ ]:
x_obs=INITIAL_X.copy()
y_obs=INITIAL_Y.copy()
# x_obs=np.append(x_obs,110.0)  # optional: add a measured point
# y_obs=np.append(y_obs,true_yield(110.0)+np.random.default_rng(SEED+99).normal(0,NOISE_STD))
plot_gp(x_obs,y_obs,ell=15,
        title=f'GP posterior with {len(x_obs)} measured reactions').show()

---

## 5. UCB chooses the next experiment · 5 min

For maximum yield, **Upper Confidence Bound** uses predicted yield and uncertainty:

$$\operatorname{UCB}(x)=\mu(x)+\kappa\sigma(x).$$

**TODO 5:** Try `kappa=0`, `1`, `4`. Which temperature does the algorithm recommend? Why?

In [ ]:
kappa=2.0 # TODO 5: try 0, 1, 4
plot_gp(x_obs,y_obs,ell=15,kappa=kappa,
        title='From predictions to the next experimental temperature').show()

---

## 6. The complete closed loop: five simulated experiments · 7 min

**Fit surrogate → compute UCB → choose temperature → measure yield → update observations → repeat.**

Only the simulator knows the true reaction curve. We exclude previously measured grid locations to avoid exact repeats.

**TODO 6:** Run with five steps and identify the one line implementing each part of the BO loop.

- Try ten steps **after class**.

In [ ]:
n_steps=5  # TODO 6: try 10 after class
kappa=2.0
ell=15.0
rng_loop=np.random.default_rng(SEED+50)
x_history=INITIAL_X.copy()
y_history=INITIAL_Y.copy()
snapshots=[]
for step in range(n_steps):
    mu,sd=gp_predict(x_history,y_history,X_GRID,ell=ell,
                       amplitude=PRIOR_SD,mean=PRIOR_MEAN,noise_std=NOISE_STD)
    acquisition=mu+kappa*sd
    used=np.any(np.isclose(X_GRID[:,None],x_history[None,:],atol=1e-8),axis=1) # check repeats
    acquisition[used]=-np.inf
    next_T=float(X_GRID[np.argmax(acquisition)])
    snapshots.append((step+1,next_T,x_history.copy(),y_history.copy()))
    next_y=float(true_yield(next_T)+rng_loop.normal(0,NOISE_STD))
    x_history=np.append(x_history,next_T)
    y_history=np.append(y_history,next_y)
    print(f'Iteration {step+1}: {next_T:.1f}°C → {next_y:.1f}% observed yield')
print(f'Best measured result: {y_history.max():.1f}% at {x_history[y_history.argmax()]:.1f}°C')
for step,next_T,x_prev,y_prev in [snapshots[0],snapshots[-1]]:
    plot_gp(x_prev,y_prev,ell=ell,kappa=kappa,
        title=f'Before experiment {step}: recommended {next_T:.1f}°C').show()
plot_gp(x_history,y_history,ell=ell,
        title='Final GP posterior after sequential experiments').show()

---

## 7. Take-home summary · 2 min

**Normal uncertainty → joint covariance → GP prior → measurements → GP posterior → UCB → next experiment → repeat.**

**Exit question:** Why can changing the kernel length scale change the next recommended experiment even if the measured yields have not changed?

*A GP is one possible probabilistic surrogate model for BO. It is not BO itself; the acquisition function and sequential decision loop complete the workflow.*

Everything below is optional after class.

---
# Optional extensions · after class

First explore alternative kernels and acquisition decisions; then increase the number of **input variables** from one (temperature) to four (temperature, time, catalyst loading, concentration). Finally, compare our handwritten BO pieces against mature packages.

**Distinction:** Two-dimensional *joint Gaussian outputs* $[f(90),f(95)]$ are not the same thing as a two-dimensional *experimental input* such as (temperature, reaction time).

**Install packages only when you reach their optional sections**

---

## A. Compare alternative kernels · after class

RBF priors are very smooth. Matérn kernels allow rougher plausible functions; a linear kernel encodes a linear trend. The same numeric length scale does not necessarily mean the same smoothness across different kernel families.

**TODO A:** Try the Matérn kernels at `ell=6` and `ell=40`. Describe the differences.

In [ ]:
t=np.linspace(60,140,100)
families=['RBF','Matern 3/2','Matern 5/2','Linear']
fig=make_subplots(rows=2,cols=2,subplot_titles=families)
for j,family in enumerate(families):
    curves=sample_prior(t,family,ell=15,amplitude=18,mean=50,n=4,seed=SEED)
    row,col=divmod(j,2)
    for i,y in enumerate(curves):
        fig.add_trace(go.Scatter(x=t,y=y,mode='lines',
                line=dict(width=2),showlegend=j==0,name=f'Curve {i+1}'),
                row=row+1,col=col+1)
fig.update_layout(title='Prior functions from different kernels',
                  height=680,template='plotly_white')
fig.show()
# A periodic kernel is also supported: sample_prior(t,'Periodic',ell=1.0).
# In this helper the periodic length scale is dimensionless, unlike RBF ell.

**Note**: RBF samples look very smooth; Matérn 3/2 allows visibly rougher functions. A linear kernel produces straight lines with uncertain intercept and slope.

---

## B. Does kernel choice change UCB? · after class

Hold the data fixed. Swap the covariance model and inspect the resulting posterior/acquisition. Be careful not to claim that one kernel is universally correct for chemical systems.

**TODO B:** Run each kernel and try length scales 6, 15 and 40.

In [ ]:
# Compare kernels using the same measured yields
for family in ['RBF','Matern 3/2','Matern 5/2']:
    mu,sd=gp_predict(INITIAL_X,INITIAL_Y,X_GRID,kind=family,ell=15,
                      amplitude=PRIOR_SD,mean=PRIOR_MEAN,noise_std=NOISE_STD)
    chosen=X_GRID[np.argmax(mu+2*sd)]
    print(f'{family:12s}: next experiment ≈ {chosen:.1f}°C')
    plot_gp(INITIAL_X,INITIAL_Y,ell=15,kind=family,kappa=2,
            title=f'UCB under the {family} kernel').show()

---

## C. Visualise two-input slices of a four-input chemistry problem · after class

Now the **experimental input** has four coordinates: temperature, reaction time, catalyst loading and concentration. We can plot a 2D slice by holding the other two variables constant.

This is a **different fabricated toy chemistry landscape**, not a direct extension of the one-input function. In general, higher-dimensional response surfaces cannot be fully displayed on one 2D graph.

**TODO C:** Change the fixed catalyst loading and concentration, and watch the yield heatmap change.

In [ ]:
def yield_4d(x):
    # Invented, smooth, synthetic chemistry-like response; not real data.
    x=np.asarray(x,dtype=float)
    T,t,cat,conc=x[...,0],x[...,1],x[...,2],x[...,3]
    return (17 + 44*np.exp(-.5*((T-114)/17)**2-.5*((t-46)/20)**2)
            +20*np.exp(-.5*((cat-.72)/.23)**2-.5*((conc-.18)/.065)**2)
            +3*np.sin((T-60)/16))

BOUNDS=np.array([[60.,10.,.1,.05],[140.,90.,1.,.30]])
NAMES=['Temperature (°C)','Reaction time (min)','Catalyst loading','Concentration (M)']
def to_physical(u):
    return BOUNDS[0]+np.asarray(u)*(BOUNDS[1]-BOUNDS[0])

tgrid=np.linspace(60,140,51)
timegrid=np.linspace(10,90,51)
TT,TM=np.meshgrid(tgrid,timegrid)
cat_fixed=.72   # TODO C: change to .25
conc_fixed=.18  # TODO C: change to .08
inputs=np.stack([TT.ravel(),TM.ravel(),
    np.full(TT.size,cat_fixed),np.full(TT.size,conc_fixed)],axis=1)
Z=yield_4d(inputs).reshape(TT.shape)
fig=go.Figure(go.Heatmap(x=tgrid,y=timegrid,z=Z,
    colorscale='Viridis',colorbar=dict(title='Yield (%)')))
fig.update_layout(title='2D slice of a 4D simulated chemical response',
    xaxis_title='Temperature (°C)',yaxis_title='Reaction time (min)',
    template='plotly_white',height=470)
fig.show()
print('Number of settings at 5 levels for each of 4 variables:',5**4)

---

## D. Use scikit-learn for a 4D GP surrogate · after class

**scikit-learn** handles GP regression, but we still choose the acquisition and search it. The four input variables are all scaled to $[0,1]$, so their different physical units do not cause a length-scale mismatch.

**TODO D:** Vary the `kappa_4d` value. Check which experimental setting is recommended. We use a finite random candidate pool, not exact continuous optimisation.

In [ ]:
# scikit-learn is often already installed in Colab.
# If not: %pip install scikit-learn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel,Matern
rng4=np.random.default_rng(SEED+100)
X_train_4d=rng4.uniform(0,1,(14,4))
y_train_4d=yield_4d(to_physical(X_train_4d))+rng4.normal(0,NOISE_STD,14)
# Standardize y manually; kernel amplitude and noise now share sensible units.
k4=ConstantKernel(1.0,constant_value_bounds='fixed')*Matern(
    length_scale=[.27]*4,length_scale_bounds='fixed',nu=2.5)
gp4=GaussianProcessRegressor(kernel=k4,alpha=(NOISE_STD/20)**2,
     optimizer=None,normalize_y=False,random_state=SEED)
gp4.fit(X_train_4d,(y_train_4d-50)/20)
candidates=rng4.uniform(0,1,(2500,4))
mean_s,sd_s=gp4.predict(candidates,return_std=True)
mean4=50+20*mean_s
sd4=20*sd_s
kappa_4d=2.0  # TODO D: try 0 and 4
index=int(np.argmax(mean4+kappa_4d*sd4))
print('4-variable UCB recommendation (approximate grid search):')
for label,value in zip(NAMES,to_physical(candidates[index])):
    print(f'  {label}: {value:.3f}')
print(f'Predicted yield = {mean4[index]:.1f}% ± {sd4[index]:.1f} (1 SD)')

---

## E. BoTorch: GP + acquisition optimizer · after class

**BoTorch** can fit a GP and optimize an acquisition function over continuous multidimensional bounds.

Doc: https://botorch.org/

First run `%pip install botorch gpytorch` in a new Colab cell. Then set `RUN_BOTORCH=True`. BoTorch's UCB uses $\sqrt{\beta}\,\sigma$, so `beta=kappa**2` matches our earlier notation.

In [ ]:
%pip install botorch gpytorch

In [ ]:
RUN_BOTORCH=False  # TODO E: set True AFTER installing botorch
if RUN_BOTORCH:
    import torch
    from botorch.models import SingleTaskGP
    from botorch.models.transforms import Normalize,Standardize
    from botorch.fit import fit_gpytorch_mll
    from gpytorch.mlls import ExactMarginalLogLikelihood
    from botorch.acquisition import UpperConfidenceBound
    from botorch.optim import optimize_acqf
    torch.manual_seed(SEED)
    train_X=torch.tensor(X_train_4d,dtype=torch.double)
    train_Y=torch.tensor(y_train_4d[:,None],dtype=torch.double)
    model=SingleTaskGP(train_X,train_Y,
                       input_transform=Normalize(d=4),
                       outcome_transform=Standardize(m=1))
    fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood,model))
    acq=UpperConfidenceBound(model=model,beta=2.0**2)
    limits=torch.stack([torch.zeros(4,dtype=torch.double),
                        torch.ones(4,dtype=torch.double)])
    new_X,_=optimize_acqf(acq_function=acq,bounds=limits,
                           q=1,num_restarts=5,raw_samples=32)
    for label,value in zip(NAMES,to_physical(new_X.detach().numpy()[0])):
        print(f'{label}: {value:.3f}')
else:
    print('BoTorch optional example skipped; install package and set flag True.')

---

## F. BayBE: a chemistry-oriented experimental campaign · after class

**BayBE** lets us declare a search space, measured target and a campaign that recommends configurations. Here we use **four discrete variables with five allowed levels each** (625 possible experiments).

Doc: https://emdgroup.github.io/baybe/stable/

First run `%pip install baybe` in a new Colab cell. Then set `RUN_BAYBE=True`. This example uses the default BayBE recommender.

In [ ]:
%pip install baybe

In [ ]:
RUN_BAYBE=True  # TODO F: set True AFTER installing baybe
if RUN_BAYBE:
    import pandas as pd
    from baybe import Campaign
    from baybe.objectives import SingleTargetObjective
    from baybe.parameters import NumericalDiscreteParameter
    from baybe.searchspace import SearchSpace
    from baybe.targets import NumericalTarget
    levels={'Temperature':np.linspace(60,140,5).tolist(),
            'Time_min':np.linspace(10,90,5).tolist(),
            'Catalyst_loading':np.linspace(.1,1,5).tolist(),
            'Concentration_M':np.linspace(.05,.30,5).tolist()}
    space=SearchSpace.from_product(parameters=[
        NumericalDiscreteParameter(name=key,values=values)
        for key,values in levels.items()])
    campaign=Campaign(searchspace=space,
        objective=SingleTargetObjective(
            target=NumericalTarget(name='Yield',mode='MAX')))
    rng_baybe=np.random.default_rng(SEED+200)
    measurements=[]
    for _ in range(10):
        row={name:float(rng_baybe.choice(vals)) for name,vals in levels.items()}
        settings=[row[k] for k in levels]
        row['Yield']=float(yield_4d(settings)+rng_baybe.normal(0,NOISE_STD))
        measurements.append(row)
    campaign.add_measurements(pd.DataFrame(measurements))
    print(campaign.recommend(batch_size=1).to_string(index=False))
else:
    print('BayBE optional example skipped; install package and set flag True.')

---
## After-class questions and references

**To test/try:**
- Explain why short length scales lead to rapid variation
- compare RBF and Matérn priors
- check how kernel and κ change UCB
- run 10 or 20 1D BO steps
- try using a second input variable
- repeat the 4D optimisation with a new random seed
- discuss practical constraints and failed experiments.

**For a real laboratory:** Decide on feasible input bounds, allowed categorical factors and safety constraints, handle measurement noise/replicates, verify units and transformations, and avoid over-interpreting a small synthetic demo. GPs on raw yield values do not automatically respect the 0–100% physical range.

**Official package documentation:**
- [scikit-learn: GaussianProcessRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html)
- [BoTorch: Getting Started](https://botorch.org/docs/getting_started)
- [BoTorch: Acquisition functions](https://botorch.readthedocs.io/en/stable/acquisition.html)
- [BayBE: Search spaces](https://emdgroup.github.io/baybe/latest/components/searchspace.html)
- [BayBE: Campaigns and recommendations](https://emdgroup.github.io/baybe/latest/concepts/getting_recommendations.html)

Note that the package APIs may change; the main notebook does not depend on BoTorch or BayBE.